# Ch 22 · Determinism and Reproducibility

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/22-determinism-and-reproducibility.ipynb)

Companion notebook for the chapter page `chapters/22-determinism-and-reproducibility.html` ([page](../chapters/22-determinism-and-reproducibility.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/22-determinism-and-reproducibility/`). Budgets grow on a GPU (`ON_GPU`) and stay small on a CPU. **This chapter is about GPU behaviour**: on Warp's CPU device threads run one after another in index order, so every CPU run is bitwise identical and several examples print trivially equal results there. Run this notebook on a GPU runtime to see the real effects. Timings printed here are your numbers; CPU numbers are never GPU numbers.

The examples run in the same order as on the chapter page (Examples 22.1 to 22.9).

In [ ]:
RESULTS = {}   # measured values, filled after each example and printed in the results table

## 1. Float atomics and slot counters
Float addition is not associative, so a sum whose terms arrive in a different order can differ in the last bits. A counter atomic (`slot = wp.atomic_add(counter, 0, 1)`) hands out slots in arrival order. Warp's `wp.DeterministicMode.RUN_TO_RUN` module option records the contributions and sorts them before reducing; MJWarp's kernel factories set it (plus `deterministic_max_records`) when ATOMICS is on. Part 2 emulates GPU arrival order in NumPy, so it shows the effect on CPU too.

In [ ]:
# Example 22.1 - ex1_atomic_order.py
"""Example 22.1 - float atomics and slot counters, with and without Warp's RUN_TO_RUN mode.

Two kernel factories, built the way MJWarp builds its deterministic kernels: when
deterministic=True they set the Warp module option "deterministic" to
wp.DeterministicMode.RUN_TO_RUN. Kernel A sums N float32 values into one accumulator with
wp.atomic_add (a scatter/reduce atomic). Kernel B reserves a slot with
slot = wp.atomic_add(counter, 0, 1) and writes its thread id there (a counter atomic, the same
pattern as contact slots in collision_core.write_contact). Each kernel runs twice per mode and we
print the bits. On a GPU the default mode can change the low bits of the sum and the slot order
between runs; RUN_TO_RUN makes them repeat. On Warp's CPU device threads run one after another in
index order and Warp compiles the deterministic path for CUDA only, so every CPU run matches.
Part 2 emulates GPU arrival order in NumPy (random permutations, sequential float32 sums) and
shows the record-sort-reduce idea behind RUN_TO_RUN.
Run: python ex1_atomic_order.py
"""
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING

ON_GPU = wp.get_device().is_cuda
N = 1 << 20 if ON_GPU else 1 << 16


def hexf(v) -> str:
    """Bit pattern of a float32 as 8 hex digits."""
    return f"{int(np.float32(v).view(np.uint32)):08x}"


def make_sum_kernel(deterministic: bool = False):
    opts = {"enable_backward": False}
    if deterministic:
        opts["deterministic"] = wp.DeterministicMode.RUN_TO_RUN

    @wp.kernel(module="unique", module_options=opts)
    def atomic_sum(x: wp.array[float], total: wp.array[float]):
        i = wp.tid()
        wp.atomic_add(total, 0, x[i])  # scatter/reduce: the return value is unused

    return atomic_sum


def make_slot_kernel(deterministic: bool = False):
    opts = {"enable_backward": False}
    if deterministic:
        opts["deterministic"] = wp.DeterministicMode.RUN_TO_RUN

    @wp.kernel(module="unique", module_options=opts)
    def reserve_slot(counter: wp.array[int], owner: wp.array[int]):
        i = wp.tid()
        slot = wp.atomic_add(counter, 0, 1)  # counter atomic: the return value IS used
        owner[slot] = i

    return reserve_slot


x_np = np.random.default_rng(0).random(N, dtype=np.float32)
x = wp.array(x_np, dtype=float)
NSLOT = 16
print(f"device: {wp.get_device()} | N = {N}")
for det in (False, True):
    mode = "RUN_TO_RUN    " if det else "NOT_GUARANTEED"
    k_sum, k_slot = make_sum_kernel(det), make_slot_kernel(det)
    sums, orders = [], []
    for _ in range(2):
        total = wp.zeros(1, dtype=float)
        wp.launch(k_sum, dim=N, inputs=[x], outputs=[total])
        counter, owner = wp.zeros(1, dtype=int), wp.zeros(NSLOT, dtype=int)
        wp.launch(k_slot, dim=NSLOT, inputs=[counter], outputs=[owner])
        sums.append(total.numpy()[0])
        orders.append(owner.numpy().tolist())
    print(f"{mode} sum run1 {sums[0]:.4f} (0x{hexf(sums[0])})  run2 0x{hexf(sums[1])}"
          f"  bitwise equal: {hexf(sums[0]) == hexf(sums[1])}")
    print(f"{mode} slot owners run1 {orders[0][:8]}...  same order in run2:"
          f" {orders[0] == orders[1]}")

# Part 2: what a GPU does to the same sum. Threads arrive in an order we do not control.
rng = np.random.default_rng(1)
vals = rng.random(4096, dtype=np.float32)
NORDER = 200
racy, sorted_red = set(), set()
for _ in range(NORDER):
    perm = rng.permutation(vals.size)            # arrival order of the 4096 "threads"
    racy.add(hexf(np.add.accumulate(vals[perm], dtype=np.float32)[-1]))
    # RUN_TO_RUN idea: keep (thread id, value) records, sort by thread id, then reduce in order
    records = np.stack([perm, vals[perm].view(np.uint32)], axis=1)
    records = records[np.argsort(records[:, 0], kind="stable")]
    sorted_red.add(hexf(np.add.accumulate(records[:, 1].astype(np.uint32).view(np.float32),
                                          dtype=np.float32)[-1]))
exact = float(np.sum(vals, dtype=np.float64))
ulp = float(np.spacing(np.float32(exact)))
r = sorted(int(h, 16) for h in racy)
print(f"emulated GPU, {NORDER} arrival orders of 4096 values (exact sum {exact:.6f}):")
print(f"  plain atomic order : {len(racy)} distinct float32 results, spread {r[-1] - r[0]} ulp"
      f" (1 ulp = {ulp:.2e})")
print(f"  record-sort-reduce : {len(sorted_red)} distinct result: 0x{next(iter(sorted_red))}")

In [ ]:
# record the measured result for the results table
k_plain, t = make_sum_kernel(False), []
for _ in range(2):
    tot = wp.zeros(1, dtype=float)
    wp.launch(k_plain, dim=N, inputs=[x], outputs=[tot])
    t.append(hexf(tot.numpy()[0]))
RESULTS["ex1"] = (("bits differ" if t[0] != t[1] else "same bits") + " in 2 NONE runs"
                  f"; emulated: {len(racy)} distinct sums")

## 2. Contact slot order
Every contact takes the next free slot of the shared pool through `wp.atomic_add(nacon_out, 0, 1)` in `collision_core.write_contact`. Compare two runs in raw slot order and in a canonical order (sorted by world, geom pair and position). On a GPU the raw order usually differs while the sorted contacts match.

In [ ]:
# Example 22.2 - ex2_contact_order.py
"""Example 22.2 - contact slot order: raw pool order vs a canonical (sorted) order.

box_stack(5) in NWORLD worlds, each world nudged differently, runs NSTEP steps twice from the
same start. Every contact takes the next free slot of the shared pool through
wp.atomic_add(nacon_out, 0, 1) (collision_core.write_contact), so on a GPU the slot order depends
on thread timing. We compare the two runs in raw slot order and after sorting every contact by
(worldid, geom pair, position). On a GPU the raw order usually differs while the sorted contacts
match; on Warp's CPU device the threads run in index order, so both orders match. The pool is
not grouped by world even on CPU: one kernel per geom-type pair fills it for all worlds.
Run: python ex2_contact_order.py [--dump-json PATH]
  --dump-json PATH writes slot worldids and geom pairs of a 2-world run (contact-order figure).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200 if ON_GPU else 40

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)


def contacts_after(nworld: int, nstep: int):
    """Run box_stack from a per-world nudged start; return the live contacts in slot order."""
    d = mjw.make_data(mjm, nworld=nworld, nconmax=32, njmax=128)
    qvel = np.zeros((nworld, mjm.nv), dtype=np.float32)
    qvel[:, 0::6] = np.linspace(0.0, 0.05, nworld)[:, None]   # each world slides a bit differently
    d.qvel.assign(qvel)
    for _ in range(nstep):
        mjw.step(m, d)
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    c = d.contact
    out = {"worldid": c.worldid.numpy()[:n], "geom": c.geom.numpy()[:n],
           "pos": c.pos.numpy()[:n], "dist": c.dist.numpy()[:n]}
    assert not d.overflow.numpy().any(), "overflow: raise nconmax/njmax"
    return out


def canonical(c):
    """Permutation that sorts contacts by (worldid, geom0, geom1, pos.x, pos.y, pos.z)."""
    p = c["pos"]
    return np.lexsort((p[:, 2], p[:, 1], p[:, 0], c["geom"][:, 1], c["geom"][:, 0], c["worldid"]))


a, b = contacts_after(NWORLD, NSTEP), contacts_after(NWORLD, NSTEP)
print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} steps"
      f" | contacts run A {len(a['dist'])}, run B {len(b['dist'])}")
same_raw = all(np.array_equal(a[k], b[k]) for k in a)
pa, pb = canonical(a), canonical(b)
same_sorted = all(np.array_equal(a[k][pa], b[k][pb]) for k in a)
print("raw slot order identical in both runs:", same_raw)
print("contacts identical after canonical sort:", same_sorted)
print("first 12 slots, worldid:", a["worldid"][:12].tolist())
print("pool grouped by world (worldid never decreases)?", bool(np.all(np.diff(a["worldid"]) >= 0)))
w0 = a["worldid"] == 0
print("world 0: slots", np.flatnonzero(w0).tolist())
print("world 0 geom pairs in slot order:", [tuple(g) for g in a["geom"][w0].tolist()])
print("contacts per world:", np.bincount(a["worldid"], minlength=NWORLD).tolist())

if args.dump_json:
    c = contacts_after(2, NSTEP)
    out = {"nworld": 2, "nstep": NSTEP, "worldid": c["worldid"].tolist(),
           "geom": c["geom"].tolist(), "canonical": canonical(c).tolist()}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(out["worldid"]), "contacts")

In [ ]:
# record the measured result for the results table
RESULTS["ex2"] = f"raw order same: {same_raw}; sorted same: {same_sorted}"

## 3. The deterministic setter
`m.opt.deterministic` is a `mjw.DeterminismType` bitmask: CONTACTS = 1, CONSTRAINT = 2, ATOMICS = 4, ALL = 7. `True`/`False`/`None` map to ALL/NONE, a complement such as `~DeterminismType.CONTACTS` is masked to the valid bits, and values outside ALL raise `ValueError`. `override_model` parses the same option from strings.

In 4.0.1 **only ATOMICS has an effect**: `collision_driver._sort_contacts` and `constraint._sort_constraints` are `pass` stubs, so ALL behaves exactly like ATOMICS.

In [ ]:
# Example 22.3 - ex3_deterministic_setter.py
"""Example 22.3 - the m.opt.deterministic setter: bools, ints, complements and bad values.

m.opt.deterministic is a DeterminismType bitmask (types.py: CONTACTS=1, CONSTRAINT=2, ATOMICS=4,
ALL=7). The Option setter maps True/False/None to ALL/NONE, masks a negative complement such as
~DeterminismType.CONTACTS to the valid bits, and raises ValueError for anything outside ALL.
io.override_model parses the same option from strings ("atomics", "~CONTACTS", "contacts|atomics").
Run: python ex3_deterministic_setter.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.io import override_model  # private API: may change between versions

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

DT = mjw.DeterminismType


def show(v: int) -> str:
    """Name the set bits of a DeterminismType value, e.g. 6 -> 'CONSTRAINT|ATOMICS'."""
    names = [f.name for f in (DT.CONTACTS, DT.CONSTRAINT, DT.ATOMICS) if v & f]
    return "|".join(names) or "NONE"


NAMES = ["NONE", "CONTACTS", "CONSTRAINT", "ATOMICS", "ALL"]
print("DeterminismType:", {n: int(getattr(DT, n)) for n in NAMES})

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
print("default after put_model:", m.opt.deterministic, show(m.opt.deterministic))

for label, value in [("True", True), ("False", False), ("None", None),
                     ("np.bool_(True)", np.bool_(True)), ("ATOMICS", DT.ATOMICS),
                     ("CONTACTS | ATOMICS", DT.CONTACTS | DT.ATOMICS),
                     ("~CONTACTS", ~DT.CONTACTS), ("-1", -1), ("7", 7)]:
    m.opt.deterministic = value
    v = m.opt.deterministic
    print(f"set {label:20s} -> {v} ({show(v)}), stored as {type(v).__name__}")

# in-place bit arithmetic works because the stored value is a plain int
m.opt.deterministic = DT.ALL
m.opt.deterministic &= ~DT.ATOMICS
print("ALL with ATOMICS cleared ->", m.opt.deterministic, show(m.opt.deterministic))

for bad in [99, 8, 1.5, "ATOMICS"]:
    try:
        m.opt.deterministic = bad
    except (TypeError, ValueError) as e:
        print(f"set {bad!r:6} -> {type(e).__name__}: {e}")
print("value unchanged after errors:", m.opt.deterministic, show(m.opt.deterministic))

# override_model (the parser behind mjwarp-testspeed -o) understands strings
# "~NAME" clears bits from the CURRENT value, so start points matter
for start, ov in [(DT.NONE, "atomics"), (DT.NONE, "contacts | atomics"), (DT.NONE, "true"),
                  (DT.NONE, "~CONTACTS"), (DT.ALL, "~CONTACTS"), (DT.ALL, "~CONTACTS|ATOMICS")]:
    m.opt.deterministic = start
    override_model(m, {"opt.deterministic": ov})  # mutates m in place and returns None
    print(f"override from {show(start):26s} with {ov!r:22s} -> {m.opt.deterministic}"
          f" ({show(m.opt.deterministic)})")

# a deterministic model steps like any other
d = mjw.make_data(mjm, nworld=2)
m.opt.deterministic = True
mjw.reset_data_keyframe(m, d, 0)
for _ in range(3):
    mjw.step(m, d)
print("3 steps with ALL, qpos world 0:", np.round(d.qpos.numpy()[0], 6))

## 4. Bitwise reproducibility without contacts
With `DisableBit.CONTACT` there is no slot order and no contact rows. ATOMICS then makes the result bitwise only when each world has at most one active limit, friction-loss or equality row, as in the PENDULUM: those rows are still reserved in racy order. The HUMANOID hits its joint limits, so its ATOMICS line is usually, but not necessarily, True. `determinism_test.py` observes equality on 2 worlds x 3 steps; that is an observation, not a guarantee. Look at the NONE rows on a GPU: they may differ.

In [ ]:
# Example 22.4 - ex4_bitwise_contact_free.py
"""Example 22.4 - bitwise reproducibility of contact-free models under DeterminismType.ATOMICS.

Without contacts there is no contact slot order and no contact rows. If, in addition, each world has
at most one active limit, friction-loss or equality row, float atomics are the only source of
run-to-run noise, and ATOMICS removes it: the double PENDULUM (no limits, no rows) is such a model.
The HUMANOID with DisableBit.CONTACT is not: it falls into its joint limits, and the limit kernels
reserve rows with the same racy wp.atomic_add(nefc_out, worldid, 1) as contacts, which ATOMICS
does not cover. We run both, NWORLD worlds with different start velocities and controls, twice per
mode, and compare qpos, qvel and qacc bit for bit.
GPU (Colab): PENDULUM with ATOMICS is bitwise equal by construction. HUMANOID with ATOMICS is
bitwise equal only if the limit rows happen to keep their order; determinism_test.py observes
equality for its contact-free scenes on 2 worlds x 3 steps, which is not a guarantee. With NONE
either model may differ in the low bits. CPU: Warp runs threads in order, so every run matches.
Run: python ex4_bitwise_contact_free.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 8
NSTEP = 200 if ON_GPU else 30
FIELDS = ("qpos", "qvel", "qacc")


def run(mjm, mode, qvel0, ctrl0):
    """One rollout from a fixed per-world start; returns copies of FIELDS."""
    m = mjw.put_model(mjm)
    m.opt.deterministic = mode
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qvel.assign(qvel0)
    d.ctrl.assign(ctrl0)
    for _ in range(NSTEP):
        mjw.step(m, d)
    return {f: getattr(d, f).numpy().copy() for f in FIELDS}


def bitwise_equal(r1, r2):
    return all(r1[f].tobytes() == r2[f].tobytes() for f in FIELDS)


humanoid = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
humanoid.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_CONTACT   # = mjw.DisableBit.CONTACT
humanoid.opt.ls_iterations = 50   # joint limits engage; 20 trips the LS_ITERATIONS warning
models = [("HUMANOID (contacts off)", humanoid),
          ("PENDULUM", mujoco.MjModel.from_xml_string(PENDULUM))]
print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} steps")
equal = {}                                     # (model, mode) -> run1 and run2 bitwise equal
for name, mjm in models:
    rng = np.random.default_rng(0)
    qvel0 = (0.1 * rng.standard_normal((NWORLD, mjm.nv))).astype(np.float32)
    ctrl0 = (0.2 * rng.standard_normal((NWORLD, mjm.nu))).astype(np.float32)
    res = {}
    for mode in (mjw.DeterminismType.NONE, mjw.DeterminismType.ATOMICS):
        r1, r2 = run(mjm, mode, qvel0, ctrl0), run(mjm, mode, qvel0, ctrl0)
        res[mode] = r1
        equal[name, mode.name] = bitwise_equal(r1, r2)
        dq = np.abs(r1["qpos"] - r2["qpos"]).max()
        print(f"{name:24s} {mode.name:7s} run1 vs run2: max |dqpos| = {dq:.3g}, "
              f"bitwise equal: {equal[name, mode.name]}")
    dq = np.abs(res[mjw.DeterminismType.NONE]["qpos"] - res[mjw.DeterminismType.ATOMICS]["qpos"])
    print(f"{name:24s} NONE vs ATOMICS: max |dqpos| = {dq.max():.3g}"
          f" (only the summation order may differ)")
    print(f"{name:24s} worlds distinct: {not np.array_equal(r1['qacc'][0], r1['qacc'][1])}")

In [ ]:
# record the measured result for the results table
RESULTS["ex4"] = "; ".join(f"{n.split()[0]} {md}: {v}" for (n, md), v in equal.items() if md == "ATOMICS")

## 5. Contact scenes and the chaos amplifier
A toppling `box_stack(5)` under NONE, ATOMICS and ALL. The example prints the differences and does not assert 0: in 4.0.1 contact scenes are not promised to be bitwise reproducible on a GPU. Part 2 starts run B one float32 ulp away and shows how quickly a contact-rich, chaotic scene amplifies that difference to tens of centimetres.

In [ ]:
# Example 22.5 - ex5_contact_scene_differences.py
"""Example 22.5 - a contact scene under NONE, ATOMICS and ALL, and how chaos amplifies low bits.

Part 1: box_stack(5) with a sideways push that topples the stack, NWORLD worlds, two runs per
mode. We print max |dqpos| between the runs and whether the raw contact slot order differed,
without asserting anything: on a GPU the slot order of contacts and constraint rows is still
racy in 4.0.1, because the CONTACTS and CONSTRAINT sorts are `pass` stubs
(collision_driver._sort_contacts, constraint._sort_constraints). So ALL behaves like ATOMICS, and
neither promises bitwise equal contact scenes. On Warp's CPU device every run matches.
Part 2 (the chaos amplifier): run B starts 1 ulp (float32) away in one coordinate, the size of
the noise a GPU reduction leaves behind. We record max |dqpos| per world over NCHAOS steps, on
NCHAOS_WORLD worlds only (the chart needs a few curves, and the per-step host copies stay small).
Run: python ex5_contact_scene_differences.py [--dump-json PATH]
  --dump-json PATH writes the per-world log10 |dqpos| curves for the chaos-amplifier chart.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200 if ON_GPU else 40
NCHAOS = 400                                   # steps for part 2 (2 s of simulated time)
NCHAOS_WORLD = 64 if ON_GPU else 8             # worlds for part 2: 2 x 400 x 64 x 35 x 4 B = 7 MB of traces

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
DT = mjw.DeterminismType
rng = np.random.default_rng(0)
qvel0 = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
qvel0[:, 24] = rng.uniform(0.8, 1.6, NWORLD)   # push the top box (body 5, dof 24 = x velocity)


def rollout(mode, nstep, qpos_nudge=None, trace=None, nworld=NWORLD):
    m = mjw.put_model(mjm)
    m.opt.deterministic = mode
    d = mjw.make_data(mjm, nworld=nworld, nconmax=32, njmax=128)
    if qpos_nudge is not None:
        d.qpos.assign(qpos_nudge)
    d.qvel.assign(qvel0[:nworld])
    for i in range(nstep):
        mjw.step(m, d)
        if trace is not None:                  # host copy every step: fine for this demo
            trace.append(d.qpos.numpy().copy())
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    order = (d.contact.worldid.numpy()[:n].copy(), d.contact.geom.numpy()[:n].copy())
    assert not d.overflow.numpy().any(), "overflow: raise nconmax/njmax"
    return d.qpos.numpy().copy(), order


print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} steps per run")
for mode in (DT.NONE, DT.ATOMICS, DT.ALL):
    qa, oa = rollout(mode, NSTEP)
    qb, ob = rollout(mode, NSTEP)
    differ = len(oa[0]) != len(ob[0]) or not all(np.array_equal(x, y) for x, y in zip(oa, ob))
    print(f"{mode.name:8s} (={int(mode)}) max |dqpos| run A vs B = {np.abs(qa - qb).max():.3g}"
          f" | contact order differed: {differ} | contacts {len(oa[0])}")

# Part 2: a 1-ulp nudge in the top box's z, the size of a reordered float32 sum
qpos1 = np.tile(mjm.qpos0, (NCHAOS_WORLD, 1)).astype(np.float32)
qpos1[:, 30] = np.nextafter(qpos1[:, 30], np.float32(1.0))    # body 5 z (qpos 28..34)
ta, tb = [], []
rollout(DT.NONE, NCHAOS, trace=ta, nworld=NCHAOS_WORLD)
rollout(DT.NONE, NCHAOS, qpos_nudge=qpos1, trace=tb, nworld=NCHAOS_WORLD)
dq = np.abs(np.array(ta) - np.array(tb)).max(axis=2)          # (NCHAOS, NCHAOS_WORLD)
print(f"1-ulp nudge: initial |dz| = {abs(float(qpos1[0, 30]) - float(np.float32(mjm.qpos0[30]))):.2e} m"
      " (vs run A's float32 start)")
for s in (1, 10, 50, 100, 200, 400):
    print(f"  after {s:3d} steps: max |dqpos| over worlds = {dq[s - 1].max():.2e}")

if args.dump_json:
    sl = list(range(0, NCHAOS, 5))
    curves = np.round(np.log10(np.maximum(dq[sl], 1e-12)), 3).T.tolist()
    out = {"nworld": NCHAOS_WORLD, "dt": 5 * float(mjm.opt.timestep), "nstep": NCHAOS, "decim": 5,
           "log10_dq": curves, "push": [round(float(v), 3) for v in qvel0[:NCHAOS_WORLD, 24]]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(sl), "frames")

In [ ]:
# record the measured result for the results table
RESULTS["ex5"] = (f"ALL max |dqpos| A vs B = {np.abs(qa - qb).max():.3g}; "
                  f"1-ulp nudge -> {dq[-1].max():.2g} m")

## 6. The replay test
The test suite's `_Replay` records the outputs of `collision` and `make_constraint` in run A and, in run B, checks inputs bitwise and outputs up to order, then overwrites B's outputs with A's. Downstream stages can then be compared bitwise under ATOMICS. On CPU we inject the GPU's behaviour with a contact shuffler (private API, `unittest.mock`).

In [ ]:
# Example 22.6 - ex6_replay_test.py
"""Example 22.6 - a replay test in the style of determinism_test.py::_Replay.

In 4.0.1 collision and make_constraint emit contacts and constraint rows in a racy order on a GPU
(the sorts are stubs), so a contact scene is not bitwise reproducible even with ATOMICS. The test
suite works around this: run A records the full Data after each of those two stages; in run B the
same stages run, the test checks that their INPUTS are bitwise equal and their float OUTPUTS are
equal up to order (sorted bit patterns), and then it overwrites B's outputs with A's recording.
Everything downstream (solver, integrator) can then be compared bitwise under ATOMICS.
On Warp's CPU device the stage order is already fixed, so we inject the GPU's behaviour: a
"shuffler" permutes the contact slots after every collision call. Run B (shuffled, no replay)
drifts away from A; run C (shuffled + replay) is bitwise equal to A.
Both stages are patched with unittest.mock, exactly like the test suite (private API).
Memory: the recording keeps a host copy of every Data array before and after each stage call,
2 stages x 2 snapshots per step, so the tape needs about (size of Data) x 4 x NSTEP bytes of host
RAM. box_stack(5) with nconmax=32, njmax=128 is about 0.04 MB per world, so the GPU budget
(NWORLD=64, NSTEP=30) needs about 0.3 GB; nworld=1024 for 100 steps would need about 16 GB.
Run: python ex6_replay_test.py [--dump-json PATH]
  --dump-json PATH writes the per-step log10 |dqpos| of run B and run C for the chaos chart.
"""
import argparse
import contextlib
import dataclasses
import json
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import collision_driver  # private API: patch targets only
from mujoco_warp._src import constraint        # private API: patch targets only


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 64 if ON_GPU else 4     # the replay tape is host RAM: (Data size) x 4 x NSTEP
NSTEP = 30 if ON_GPU else 60
PHYSICS = ("qpos", "qvel", "qacc", "qacc_smooth", "qfrc_constraint", "qfrc_bias", "cacc")


def arrays_of(obj, prefix="d"):
    """All warp arrays reachable from a Data (nested dataclasses), keyed by path."""
    if isinstance(obj, wp.array):
        return {prefix: obj}
    if not dataclasses.is_dataclass(obj):
        return {}
    out = {}
    for name, value in vars(obj).items():
        out.update(arrays_of(value, f"{prefix}.{name}"))
    return out


def snap(arrs):
    return {k: a.numpy().copy() for k, a in arrs.items()}


def sorted_bits(a):
    """Bit patterns in sorted order: equal for two arrays that hold the same values in any order."""
    return np.sort(a.reshape(-1).view(f"u{a.itemsize}"))


class Replay:
    """Record stage outputs in the first run; check and replay them in later runs."""

    def __init__(self, d):
        self.d, self.calls, self.recording, self.i = d, [], True, 0
        self.checked = 0

    def wrap(self, stage, fn):
        def wrapper(*a, **kw):
            arrs = arrays_of(self.d)
            pre = snap(arrs)
            fn(*a, **kw)
            post = snap(arrs)
            if self.recording:
                self.calls.append((stage, pre, post))
                return
            ref_stage, ref_pre, ref_post = self.calls[self.i]
            self.i += 1
            assert stage == ref_stage, (stage, ref_stage)
            bad_in = [k for k in pre if pre[k].tobytes() != ref_pre[k].tobytes()]
            assert not bad_in, f"inputs of {stage} differ: {bad_in[:3]}"
            floats = [k for k, v in post.items() if v.dtype.kind == "f"]
            bad_out = [k for k in floats
                       if sorted_bits(post[k]).tobytes() != sorted_bits(ref_post[k]).tobytes()]
            assert not bad_out, f"float outputs of {stage} differ beyond order: {bad_out[:3]}"
            for k, arr in arrs.items():                # replay: restore run A's exact outputs
                if arr.size:
                    arr.assign(ref_post[k])
            self.checked += 1
        return wrapper

    @contextlib.contextmanager
    def patch(self):
        self.recording, self.i = not self.calls, 0
        with mock.patch.object(collision_driver, "collision",
                               self.wrap("collision", collision_driver.collision)), \
             mock.patch.object(constraint, "make_constraint",
                               self.wrap("make_constraint", constraint.make_constraint)):
            yield


@contextlib.contextmanager
def shuffled_contacts(d, seed):
    """Emulate the GPU: permute the live contact slots after every collision call."""
    rng, inner = np.random.default_rng(seed), collision_driver.collision

    def collision(*a, **kw):
        out = inner(*a, **kw)
        n = min(int(d.nacon.numpy()[0]), d.naconmax)
        perm = rng.permutation(n)
        for f in dataclasses.fields(d.contact):
            arr = getattr(d.contact, f.name)
            if arr.shape and arr.shape[0] == d.naconmax and n:   # skip (0,)-shaped flex fields
                v = arr.numpy()
                v[:n] = v[:n][perm]
                arr.assign(v)
        return out

    with mock.patch.object(collision_driver, "collision", collision):
        yield


mjm = mujoco.MjModel.from_xml_string(box_stack(5))
m = mjw.put_model(mjm)
m.opt.deterministic = mjw.DeterminismType.ATOMICS
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
qvel0 = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
qvel0[:, 24] = np.linspace(0.8, 1.6, NWORLD)              # topple the stack, differently per world
init = snap(arrays_of(d))
init["d.qvel"] = qvel0
DATA_MB = sum(v.nbytes for v in init.values()) / 1e6


def run(label, replay=None, shuffle_seed=None, trace=None):
    for k, arr in arrays_of(d).items():
        if arr.size:
            arr.assign(init[k])
    with contextlib.ExitStack() as st:
        if shuffle_seed is not None:
            st.enter_context(shuffled_contacts(d, shuffle_seed))
        if replay is not None:                 # entered last: wraps the shuffled collision
            st.enter_context(replay.patch())
        for _ in range(NSTEP):
            mjw.step(m, d)
            if trace is not None:
                trace.append(d.qpos.numpy().copy())
    assert not d.overflow.numpy().any(), "overflow: raise nconmax/njmax"
    return {f: getattr(d, f).numpy().copy() for f in PHYSICS}


def compare(r1, r2):
    same = [f for f in PHYSICS if r1[f].tobytes() == r2[f].tobytes()]
    return np.abs(r1["qpos"] - r2["qpos"]).max(), len(same) == len(PHYSICS)


rep = Replay(d)
ta, tb, tc = [], [], []
ref = run("A", replay=rep, trace=ta)
print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} steps | ATOMICS"
      f" | tape = {DATA_MB:.2f} MB Data x 4 x {NSTEP} = {DATA_MB * 4 * NSTEP / 1e3:.2f} GB host RAM")
print(f"run A (record)           : recorded {len(rep.calls)} stage calls"
      f" ({sum(c[0] == 'collision' for c in rep.calls)} collision,"
      f" {sum(c[0] == 'make_constraint' for c in rep.calls)} make_constraint)")
r = run("A2")
dq, same = compare(ref, r)
print(f"run A2 (plain rerun)     : max |dqpos| = {dq:.3g}, bitwise equal to A: {same}")
r = run("B", shuffle_seed=1, trace=tb)
dq, same = compare(ref, r)
print(f"run B (shuffled slots)   : max |dqpos| = {dq:.3g}, bitwise equal to A: {same}")
r = run("C", replay=rep, shuffle_seed=1, trace=tc)
dq, same = compare(ref, r)
print(f"run C (shuffled + replay): max |dqpos| = {dq:.3g}, bitwise equal to A: {same},"
      f" stage checks passed: {rep.checked}")

dqb = np.abs(np.array(ta) - np.array(tb)).max(axis=(1, 2))
dqc = np.abs(np.array(ta) - np.array(tc)).max(axis=(1, 2))
first = int(np.argmax(dqb > 0)) + 1 if (dqb > 0).any() else None
print(f"shuffled run first differs at step {first} (by {dqb[first - 1] if first else 0:.1e});"
      f" |dqpos| after 10/30/{NSTEP} steps: {dqb[9]:.1e} / {dqb[29]:.1e} / {dqb[-1]:.1e}")

if args.dump_json:
    lg = lambda a: [round(float(np.log10(max(x, 1e-12))), 3) for x in a]
    out = {"nworld": NWORLD, "nstep": NSTEP, "dt": float(mjm.opt.timestep),
           "log10_dq_shuffled": lg(dqb), "log10_dq_replayed": lg(dqc)}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, NSTEP, "frames")

In [ ]:
# record the measured result for the results table
RESULTS["ex6"] = f"run C bitwise equal: {same} ({rep.checked} stage checks)"

## 7. A reusable bitwise comparator
`compare_runs(d1, d2, fields)` compares bit patterns, not values, and reports the first differing field, world and element in hex. The exercises use the same idea.

In [ ]:
# Example 22.7 - ex7_compare_runs.py
"""Example 22.7 - compare_runs: find the first field and world where two runs differ bitwise.

A reusable helper for reproducibility tests (and the exercises). It compares the bit patterns of
the chosen Data fields, not their values, so it catches -0.0 vs 0.0 (which == calls equal) and
treats identical NaNs as equal (== flags every NaN as different), and it reports the first differing field (in the order given), world and element with
both values in hex. Demo: two identical runs of the double pendulum, then a run where world 3
starts 1e-6 rad away and world 5 starts 1 ulp (float32) away in qpos. compare_runs reports
the first difference and lists every world that differs.
Run: python ex7_compare_runs.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

FIELDS = ("qpos", "qvel", "qacc", "act", "sensordata", "energy", "time")


def compare_runs(d1, d2, fields=FIELDS):
    """Return None if every field is bitwise equal, else a dict describing the first difference.

    Fields are (nworld, ...) arrays of mjw.Data; vector dtypes (vec3, ...) are flattened per world.
    """
    for name in fields:
        a, b = getattr(d1, name).numpy(), getattr(d2, name).numpy()
        if a.shape != b.shape:
            return {"field": name, "reason": f"shape {a.shape} != {b.shape}"}
        if a.size == 0:
            continue
        ua = np.ascontiguousarray(a).reshape(a.shape[0], -1).view(f"u{a.itemsize}")
        ub = np.ascontiguousarray(b).reshape(b.shape[0], -1).view(f"u{b.itemsize}")
        ne = ua != ub
        if ne.any():
            world, idx = (int(i) for i in np.argwhere(ne)[0])
            va, vb = a.reshape(a.shape[0], -1)[world, idx], b.reshape(b.shape[0], -1)[world, idx]
            hexw = 2 * a.itemsize
            return {"field": name, "world": world, "index": idx,
                    "a": f"{float(va):.9g} (0x{int(ua[world, idx]):0{hexw}x})",
                    "b": f"{float(vb):.9g} (0x{int(ub[world, idx]):0{hexw}x})",
                    "n_elements": int(ne.sum()), "worlds": np.flatnonzero(ne.any(axis=1)).tolist()}
    return None


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)


def run(nudge=False):
    d = mjw.make_data(mjm, nworld=NWORLD)
    mjw.reset_data_keyframe(m, d, 0)
    qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
    qvel[:, 0] = np.linspace(0.0, 1.0, NWORLD)
    d.qvel.assign(qvel)
    if nudge:
        qpos = d.qpos.numpy()
        qpos[3, 0] += np.float32(1e-6)
        qpos[5, 0] = np.nextafter(qpos[5, 0], np.float32(10.0))   # the smallest float32 step
        d.qpos.assign(qpos)
    for _ in range(NSTEP):
        mjw.step(m, d)
    return d


d1, d2 = run(), run()
print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} steps")
print("identical runs  :", compare_runs(d1, d2) or "bitwise equal in " + ", ".join(FIELDS))
d3 = run(nudge=True)
diff = compare_runs(d1, d3)
print("nudged run      :")
for k, v in diff.items():
    print(f"  {k:10s} {v}")
dq = np.abs(d1.qpos.numpy() - d3.qpos.numpy()).max(axis=1)
print("max |dqpos| per world:", [f"{v:.1e}" for v in dq])

In [ ]:
# record the measured result for the results table
RESULTS["ex7"] = f"first: {diff['field']}, world {diff['world']}" if diff else "no difference"

## 8. Bitwise restarts need the warmstart
Snapshot at step 100 with `mjw.get_state`, run 100 steps, restore with `mjw.set_state`, rerun. `QPOS|QVEL` and `PHYSICS` leave `d.qacc_warmstart` at the end-of-reference value, so the solver starts elsewhere and the replay is off in the low bits. `PHYSICS | WARMSTART` is exact on CPU. Always allocate the buffer with `mujoco.mj_stateSize` (get_state does not check its size).

In [ ]:
# Example 22.8 - ex8_snapshot_replay.py
"""Example 22.8 - a reproducible run needs a complete snapshot: State.PHYSICS | State.WARMSTART.

box_stack(5) settles for NPRE steps. We snapshot with mjw.get_state, run NPOST steps (reference),
restore with mjw.set_state and run the same NPOST steps again. With QPOS|QVEL or PHYSICS the
replay starts from a different solver warmstart (d.qacc_warmstart keeps the END value of the
reference run), so the Newton solver converges to a slightly different qacc and the replay is
off in the low bits. Adding WARMSTART makes the replay exact on CPU. On a GPU the contact order
is racy, so even a complete snapshot is bitwise only up to the noise that ATOMICS cannot remove:
the notebook prints both modes so you can see your GPU's numbers.
Run: python ex8_snapshot_replay.py [--dump-json PATH]
  --dump-json PATH writes the table (mode, signature, size, max |dqpos|) for the page widget.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NPRE, NPOST = 100, 100
S = mjw.State

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
rows = []
print(f"device: {wp.get_device()} | nworld {NWORLD}"
      f" | snapshot at step {NPRE}, replay {NPOST} steps")
for mode in (mjw.DeterminismType.NONE, mjw.DeterminismType.ATOMICS):
    m = mjw.put_model(mjm)
    m.opt.deterministic = mode
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
    qvel[:, 24] = np.linspace(0.0, 0.3, NWORLD)     # a small push on the top box, per world
    d.qvel.assign(qvel)
    for _ in range(NPRE):
        mjw.step(m, d)
    full = wp.zeros((NWORLD, mujoco.mj_stateSize(mjm, int(S.INTEGRATION))), dtype=float)
    mjw.get_state(m, d, full, int(S.INTEGRATION))  # so every signature starts from the same point
    for name, sig in [("QPOS|QVEL", S.QPOS | S.QVEL), ("PHYSICS", S.PHYSICS),
                      ("PHYSICS|WARMSTART", S.PHYSICS | S.WARMSTART)]:
        mjw.set_state(m, d, full, int(S.INTEGRATION))
        size = mujoco.mj_stateSize(mjm, int(sig))
        buf = wp.zeros((NWORLD, size), dtype=float)   # exact size: get_state does not check it
        mjw.get_state(m, d, buf, int(sig))
        for _ in range(NPOST):
            mjw.step(m, d)
        ref = d.qpos.numpy().copy()
        mjw.set_state(m, d, buf, int(sig))
        for _ in range(NPOST):
            mjw.step(m, d)
        dq = np.abs(d.qpos.numpy() - ref).max()
        exact = d.qpos.numpy().tobytes() == ref.tobytes()
        print(f"{mode.name:7s} {name:18s} size {size:3d}: max |dqpos| = {dq:.3g}  bitwise: {exact}")
        rows.append({"mode": mode.name, "sig": name, "size": size, "dq": float(dq), "exact": exact})
    ovf = d.overflow.numpy()
    print(f"{mode.name:7s} overflow:", "none" if not ovf.any() else ovf)

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "npre": NPRE, "npost": NPOST, "rows": rows}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# record the measured result for the results table
r8 = next(r for r in rows if r["mode"] == "ATOMICS" and r["sig"] == "PHYSICS|WARMSTART")
RESULTS["ex8"] = f"bitwise: {r8['exact']}, max |dqpos| = {r8['dq']:.3g}"

## 9. What determinism costs
Steps/s for NONE, ATOMICS and ALL, plus the number of kernel launches per step that run in RUN_TO_RUN mode. On a GPU, RUN_TO_RUN kernels record and sort their atomic contributions, so expect them to be slower; on CPU the deterministic path is compiled out and timings differ by noise. On Colab you can also run `mjwarp-testspeed <xml> -o "opt.deterministic=ATOMICS"`.

In [ ]:
# Example 22.9 - ex9_determinism_cost.py
"""Example 22.9 - what determinism costs: steps/s for NONE, ATOMICS and ALL.

For two scenes (box_stack(5) with contacts, HUMANOID with contacts off) we time NSTEP steps of
NWORLD worlds per mode, after a warm-up step that JIT-compiles the mode's kernels. We also count,
for one sampled step, how many kernel launches run in Warp's RUN_TO_RUN mode (the same launch spy that
determinism_test.py::_record_launches uses). The counts belong to that one step: the solver loop
runs until every world converges, so its launches grow with solver_niter (printed next to the
counts), and a few launch paths branch on wp.get_device().is_cuda, so a GPU can count differently.
In 4.0.1 ALL costs the same as ATOMICS, because the
CONTACTS and CONSTRAINT sorts are stubs. On a GPU, RUN_TO_RUN kernels record their atomic
contributions and sort them before reducing, so expect them to be slower; on Warp's CPU device the
deterministic code path is compiled out (threads already run in order), so CPU timings differ by
noise only. CPU numbers are from a shared CPU dev box and say nothing about GPU speed.
Run: python ex9_determinism_cost.py [--dump-json PATH]   (on Colab you can also use:
  mjwarp-testspeed <xml> -o "opt.deterministic=ATOMICS")
  --dump-json PATH writes launches per step and RUN_TO_RUN launches per scene and mode.
"""
import argparse
import contextlib
import json
import time
from unittest import mock

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 500 if ON_GPU else 20
DT = mjw.DeterminismType


@contextlib.contextmanager
def count_launches():
    """Count kernel launches, and those compiled with the RUN_TO_RUN module option."""
    counts = {"all": 0, "run_to_run": 0}

    def wrap(launch):
        def wrapper(*a, **kw):
            kernel = kw["kernel"] if "kernel" in kw else a[0]
            counts["all"] += 1
            mode = kernel.module.options.get("deterministic", wp.DeterministicMode.NOT_GUARANTEED)
            counts["run_to_run"] += mode == wp.DeterministicMode.RUN_TO_RUN
            return launch(*a, **kw)
        return wrapper

    with mock.patch.object(wp, "launch", wrap(wp.launch)), \
         mock.patch.object(wp, "launch_tiled", wrap(wp.launch_tiled)):
        yield counts


humanoid = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
humanoid.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_CONTACT
humanoid.opt.ls_iterations = 50
scenes = [("box_stack(5)", mujoco.MjModel.from_xml_string(box_stack(5)),
           dict(nconmax=32, njmax=128)),
          ("HUMANOID, no contacts", humanoid, {})]
print(f"device: {wp.get_device()} | nworld {NWORLD} | {NSTEP} timed steps"
      + ("" if ON_GPU else " | CPU dev box: timings are noise-level, not GPU numbers"))
print(f"{'scene':22s} {'mode':8s} {'world-steps/s':>14s} {'vs NONE':>8s} {'launches/step':>14s}"
      f" {'RUN_TO_RUN':>11s} {'niter':>6s}")
rows, rates = [], {}
for name, mjm, sizes in scenes:
    base = None
    for mode in (DT.NONE, DT.ATOMICS, DT.ALL):
        m = mjw.put_model(mjm)
        m.opt.deterministic = mode
        d = mjw.make_data(mjm, nworld=NWORLD, **sizes)
        mjw.step(m, d)                         # warm-up: compiles this mode's kernels
        with count_launches() as counts:
            mjw.step(m, d)
        niter = int(d.solver_niter.numpy().max())  # solver iterations in the counted step
        wp.synchronize()
        t0 = time.perf_counter()
        for _ in range(NSTEP):
            mjw.step(m, d)
        wp.synchronize()
        rate = NWORLD * NSTEP / (time.perf_counter() - t0)
        base = base or rate
        rates[name, mode.name] = rate
        print(f"{name:22s} {mode.name:8s} {rate:14,.0f} {rate / base:8.2f} {counts['all']:14d}"
              f" {counts['run_to_run']:11d} {niter:6d}")
        assert not d.overflow.numpy().any(), "overflow"
        rows.append({"scene": name, "mode": mode.name, "launches": counts["all"],
                     "run_to_run": counts["run_to_run"], "solver_niter": niter})

if args.dump_json:   # launch counts only: timings are machine-specific
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "rows": rows}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# record the measured result for the results table
RESULTS["ex9"] = "; ".join(f"{n.split()[0].rstrip(',')} ALL/ATOMICS = {rates[n, 'ALL'] / rates[n, 'ATOMICS']:.2f}"
                           for n in dict.fromkeys(k[0] for k in rates))

## Results table
What this runtime measured in the examples above, next to what to expect on a GPU.

In [ ]:
# Results table: what this runtime measured (filled by the record cells after each example),
# next to what to expect on a GPU. On a CPU every run is bitwise identical, so the measured
# column says "same" almost everywhere; run on a GPU to see the real effects.
rows = [
    ("ex1", "Float atomics change low bits between runs (NONE)?", "GPU: often; CPU: never"),
    ("ex2", "Raw contact slot order repeats?", "GPU: usually not; CPU: yes"),
    ("ex4", "Contact-free models bitwise under ATOMICS?", "PENDULUM: yes; HUMANOID: usually (limit rows racy)"),
    ("ex5", "box_stack(5) bitwise under ALL?", "not promised in 4.0.1 (sorts are stubs)"),
    ("ex6", "Replayed run bitwise equal to run A?", "yes, under ATOMICS"),
    ("ex7", "compare_runs finds the nudged world?", "yes: qpos, world 3"),
    ("ex8", "PHYSICS|WARMSTART restore exact (ATOMICS)?", "CPU: yes; GPU: up to contact-order noise"),
    ("ex9", "ALL costs more than ATOMICS?", "no: ALL = ATOMICS in 4.0.1"),
]
print("device:", wp.get_device())
print(f"{'question':<52} {'measured here':<60} expected")
for key, q, expected in rows:
    print(f"{q:<52} {RESULTS.get(key, '(cell not run)'):<60} {expected}")

## Exercise 22.1: find the first bitwise difference
Write `first_difference(d1, d2, fields)` that returns `(field, world)` of the first bitwise difference, or `None`. View the arrays as unsigned integers of the same width.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco><option timestep="0.01"/><worldbody>
  <body pos="0 0 1.5"><joint name="h1" type="hinge" axis="0 1 0"/>
    <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
    <body pos="0 0 -.5"><joint name="h2" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/></body></body>
</worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)


def run(nudge=0.0, nworld=4, nstep=50):
    d = mjw.make_data(mjm, nworld=nworld)
    qpos = np.tile([1.2, 0.0], (nworld, 1)).astype(np.float32)
    qpos[2, 0] += nudge                       # only world 2 is nudged
    d.qpos.assign(qpos)
    for _ in range(nstep):
        mjw.step(m, d)
    return d


def first_difference(d1, d2, fields=("qpos", "qvel", "qacc")):
    ...  # your code


assert first_difference(run(), run()) is None
assert first_difference(run(), run(nudge=1e-5)) == ("qpos", 2)
print("Exercise 22.1 passed")

In [ ]:
#@title Exercise 22.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco><option timestep="0.01"/><worldbody>
  <body pos="0 0 1.5"><joint name="h1" type="hinge" axis="0 1 0"/>
    <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
    <body pos="0 0 -.5"><joint name="h2" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/></body></body>
</worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)


def run(nudge=0.0, nworld=4, nstep=50):
    d = mjw.make_data(mjm, nworld=nworld)
    qpos = np.tile([1.2, 0.0], (nworld, 1)).astype(np.float32)
    qpos[2, 0] += nudge                       # only world 2 is nudged
    d.qpos.assign(qpos)
    for _ in range(nstep):
        mjw.step(m, d)
    return d


def first_difference(d1, d2, fields=("qpos", "qvel", "qacc")):
    """(field, world) of the first bitwise difference, or None."""
    for name in fields:
        a, b = getattr(d1, name).numpy(), getattr(d2, name).numpy()
        ua = a.reshape(a.shape[0], -1).view(f"u{a.itemsize}")
        ub = b.reshape(b.shape[0], -1).view(f"u{b.itemsize}")
        bad = np.flatnonzero((ua != ub).any(axis=1))
        if bad.size:
            return name, int(bad[0])
    return None


assert first_difference(run(), run()) is None
assert first_difference(run(), run(nudge=1e-5)) == ("qpos", 2)
print("Exercise 22.1 passed")

## Exercise 22.2: which snapshot replays exactly?
Two boxes have landed (contacts and warmstart active). Assert that `PHYSICS | WARMSTART` reproduces the run exactly on CPU while `QPOS | QVEL` does not. The cell runs on CPU (`wp.ScopedDevice("cpu")`); on a GPU, try it without the scope as a stretch.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TWO_BOXES = """
<mujoco><option timestep="0.005"/><worldbody>
  <geom type="plane" size="5 5 .1"/>
  <body pos="0 0 0.3" euler="10 20 0"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
  <body pos="0.05 0 0.6" euler="0 15 30"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
</worldbody></mujoco>"""
with wp.ScopedDevice("cpu"):  # the exact-replay claim is a CPU claim
    mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)
    for _ in range(150):
        mjw.step(m, d)
    S = mjw.State

    def replay_is_exact(sig, nstep=30):
        ...  # your code: get_state, step, set_state, step, compare

    assert replay_is_exact(S.PHYSICS | S.WARMSTART)
    assert not replay_is_exact(S.QPOS | S.QVEL)
    print("Exercise 22.2 passed")

In [ ]:
#@title Exercise 22.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TWO_BOXES = """
<mujoco><option timestep="0.005"/><worldbody>
  <geom type="plane" size="5 5 .1"/>
  <body pos="0 0 0.3" euler="10 20 0"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
  <body pos="0.05 0 0.6" euler="0 15 30"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
</worldbody></mujoco>"""
with wp.ScopedDevice("cpu"):  # the exact-replay claim is a CPU claim
    mjm = mujoco.MjModel.from_xml_string(TWO_BOXES)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=4, nconmax=16, njmax=64)
    for _ in range(150):
        mjw.step(m, d)
    S = mjw.State

    def replay_is_exact(sig, nstep=30):
        """Snapshot with `sig`, run nstep (reference), restore, rerun; True if qpos is bitwise equal."""
        size = mujoco.mj_stateSize(mjm, int(sig))
        buf = wp.zeros((d.nworld, size), dtype=float)
        mjw.get_state(m, d, buf, int(sig))
        for _ in range(nstep):
            mjw.step(m, d)
        ref = d.qpos.numpy().copy()
        mjw.set_state(m, d, buf, int(sig))
        for _ in range(nstep):
            mjw.step(m, d)
        return d.qpos.numpy().tobytes() == ref.tobytes()

    assert replay_is_exact(S.PHYSICS | S.WARMSTART)
    assert not replay_is_exact(S.QPOS | S.QVEL)
    print("Exercise 22.2 passed")

## Exercise 22.3: canonical contact order
Run B holds the same contacts as run A in a shuffled order. Write the permutation that sorts contacts by `(worldid, geom0, geom1, pos.x, pos.y, pos.z)`. Pure NumPy.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 40
worldid = rng.integers(0, 4, n)
geom = np.sort(rng.integers(0, 6, (n, 2)), axis=1)
pos = rng.random((n, 3)).astype(np.float32)
perm = rng.permutation(n)                          # "run B": same contacts, other slots
b = {"worldid": worldid[perm], "geom": geom[perm], "pos": pos[perm]}
a = {"worldid": worldid, "geom": geom, "pos": pos}


def canonical(c):
    ...  # your code: return an index array


assert not np.array_equal(a["geom"], b["geom"])
pa, pb = canonical(a), canonical(b)
for k in a:
    assert np.array_equal(a[k][pa], b[k][pb]), k
assert np.all(np.diff(a["worldid"][pa]) >= 0)
print("Exercise 22.3 passed")

In [ ]:
#@title Exercise 22.3: solution (double-click to show the code)
import numpy as np

rng = np.random.default_rng(0)
n = 40
worldid = rng.integers(0, 4, n)
geom = np.sort(rng.integers(0, 6, (n, 2)), axis=1)
pos = rng.random((n, 3)).astype(np.float32)
perm = rng.permutation(n)                          # "run B": same contacts, other slots
b = {"worldid": worldid[perm], "geom": geom[perm], "pos": pos[perm]}
a = {"worldid": worldid, "geom": geom, "pos": pos}


def canonical(c):
    p = c["pos"]
    return np.lexsort((p[:, 2], p[:, 1], p[:, 0], c["geom"][:, 1], c["geom"][:, 0], c["worldid"]))


assert not np.array_equal(a["geom"], b["geom"])
pa, pb = canonical(a), canonical(b)
for k in a:
    assert np.array_equal(a[k][pa], b[k][pb]), k
assert np.all(np.diff(a["worldid"][pa]) >= 0)
print("Exercise 22.3 passed")

## Stretch: extend the replay test to sensors
Add a `touch` sensor on the top box, a `framepos` of the top box and a `contact` sensor between box 3 and box 4 to the `box_stack` of Example 22.6. Extend `PHYSICS` with `sensordata` and check: (a) with replay, all sensors are bitwise equal to run A; (b) in the shuffled run without replay, which sensors differ first, and by how much? (c) Why does the test suite exclude contact sensors from its NONE-vs-ATOMICS tolerance check? On a GPU, run the test without the shuffler and compare.

In [ ]:
# your sensor replay test here